In [ ]:
import os
import yaml
from ultralytics import YOLO, settings
import torch
import cv2
import matplotlib.pyplot as plt
import random
from sahi.predict import get_sliced_prediction
from sahi import AutoDetectionModel
from sahi.utils.cv import visualize_object_predictions
import numpy as np

settings.update({'datasets_dir': r'E:\Projects\CCTV_horkang'})
print("Settings updated to project directory.")

Settings updated to project directory.


In [13]:
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    # Verify it sees the 5060 and the 12.0 capability
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")
    
    # Critical test: perform a calculation on the GPU
    # If this doesn't crash, your 5060 is ready!
    test_tensor = torch.randn(1024, 1024).cuda()
    result = test_tensor @ test_tensor
    print("✅ GPU calculation successful!")

CUDA available: True
GPU: NVIDIA GeForce RTX 5060
Compute Capability: (12, 0)
✅ GPU calculation successful!


In [14]:
device = '0' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if device=='0' else 'CPU'})")

Using device: 0 (NVIDIA GeForce RTX 5060)


In [4]:
project_root = r'E:\Projects\CCTV_horkang\train_dataset_v26'
train_images = os.path.join(project_root, 'train', 'images')
val_images = os.path.join(project_root, 'valid', 'images')

data_config = {
    'train': train_images,
    'val': val_images,
    'nc': 1,
    'names': ['Person']
}

yaml_path = os.path.join(project_root, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)


In [5]:
import os
img_path = r'E:\Projects\CCTV_horkang\train_dataset_v26\train\images'
total_images = sum([len(files) for r, d, files in os.walk(img_path) if any(f.endswith('.jpg') for f in files)])
print(f"Total images found in folder: {total_images}")

Total images found in folder: 1127


In [7]:
model = YOLO('yolo26n.pt')
model.train(
    data=r'E:\Projects\CCTV_horkang\train_dataset_v26\data.yaml',
    epochs= 100,      # High epochs for better accuracy
    imgsz=640,       # Standard resolution
    batch=8,        # Adjust to 8 or 4 if you get "Out of Memory"
    device=device,   # Uses your NVIDIA GPU
    workers=4,       # Number of CPU threads loading data
    name='human_detector_v5',
    exist_ok=True,
    patience=20       # Early stopping patience
)

New https://pypi.org/project/ultralytics/8.4.48 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.10.19 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\Projects\CCTV_horkang\train_dataset_v26\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosa

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000027F608931F0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [15]:
detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=r'runs/detect/human_detector_v5/weights/best.pt',
    confidence_threshold=0.4,
    device='cuda:0'           
)

In [ ]:
test_folder = r'E:\Projects\CCTV_horkang\sample_data\channel_9 10s'
output_dir = r'E:\Projects\CCTV_horkang\runs\detect\predict'
os.makedirs(output_dir, exist_ok=True)

all_images = [f for f in os.listdir(test_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
selected_images = random.sample(all_images, min(10, len(all_images)))

print(f"random {len(selected_images)}  Channel 9")

for img_name in selected_images:
    image_path = os.path.join(test_folder, img_name)
    
    results = get_sliced_prediction(
        image_path,
        detection_model,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        verbose=0
    )

    prediction_visual = visualize_object_predictions(
        image=np.array(results.image),
        object_prediction_list=results.object_prediction_list
    )

    output_image = cv2.cvtColor(prediction_visual['image'], cv2.COLOR_RGB2BGR)
    save_path = os.path.join(output_dir, f"pred_{img_name}")
    cv2.imwrite(save_path, output_image)

    plt.figure(figsize=(12, 7))
    plt.imshow(prediction_visual['image'])
    plt.title(f"Predict: {img_name} (Found: {len(results.object_prediction_list)} persons)")
    plt.axis('off')
    plt.show()

random 10  Channel 9


<Figure size 1200x700 with 1 Axes>

<Figure size 1200x700 with 1 Axes>

<Figure size 1200x700 with 1 Axes>

<Figure size 1200x700 with 1 Axes>

...

<Figure size 1200x700 with 1 Axes>

<Figure size 1200x700 with 1 Axes>

<Figure size 1200x700 with 1 Axes>

<Figure size 1200x700 with 1 Axes>

<Figure size 1200x700 with 1 Axes>